# Korean EditTagger Training (Colab)

이 노트북은 `edit_tagger_dataset.tgz`와 `train_edit_tagger.py`를 Google Drive에 올려둔 상태를 기준으로, Colab free tier에서 `KoELECTRA` 기반 EditTagger를 학습합니다.

## 0. 런타임 확인

- `런타임 > 런타임 유형 변경 > GPU`로 먼저 바꾸세요.
- Drive 경로는 `MyDrive/grammarly_korean/` 기준입니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls -lah /content/drive/MyDrive/grammarly_korean

In [ ]:
!mkdir -p /content/work
!cp /content/drive/MyDrive/grammarly_korean/edit_tagger_dataset.tgz /content/work/
!cp /content/drive/MyDrive/grammarly_korean/train_edit_tagger.py /content/work/
!ls -lah /content/work

In [ ]:
!tar -xzf /content/work/edit_tagger_dataset.tgz -C /content/work
!ls -lah /content/work/edit_tagger_dataset
!wc -l /content/work/edit_tagger_dataset/train.jsonl /content/work/edit_tagger_dataset/validation.jsonl

In [ ]:
!pip install -q transformers datasets accelerate seqeval optimum onnx onnxruntime

In [ ]:
!nvidia-smi

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
OUTPUT_DIR = '/content/drive/MyDrive/grammarly_korean/checkpoints/edit_tagger_v1'
TRAIN_LIMIT = 50000
VALID_LIMIT = 5000
BATCH_SIZE = 8
GRAD_ACCUM = 4
EPOCHS = 2
OUTPUT_DIR

## 1차 학습 권장값

free tier에서는 먼저 아래 설정으로 검증하세요.

- `TRAIN_LIMIT = 50000`
- `VALID_LIMIT = 5000`
- `BATCH_SIZE = 8`
- `GRAD_ACCUM = 4`
- `EPOCHS = 2`

In [ ]:
!python /content/work/train_edit_tagger.py \
  --dataset-dir /content/work/edit_tagger_dataset \
  --output-dir "$OUTPUT_DIR" \
  --train-limit $TRAIN_LIMIT \
  --validation-limit $VALID_LIMIT \
  --batch-size $BATCH_SIZE \
  --grad-accum $GRAD_ACCUM \
  --epochs $EPOCHS \
  --save-steps 500 \
  --eval-steps 500 \
  --logging-steps 100

In [ ]:
!ls -lah "$OUTPUT_DIR"
!cat "$OUTPUT_DIR/metrics.json"

## 중요 지표

학습이 끝나면 `metrics.json`에서 아래를 먼저 보세요.

- `non_keep_f1`
- `non_keep_recall`

이 둘이 좋아야 `KEEP만 찍는 문제`를 줄일 수 있습니다.

## 2차 실험

1차가 안정적으로 돌면 아래처럼 범위를 늘리세요.

- `TRAIN_LIMIT = 120000` 또는 `0`(전체)
- `VALID_LIMIT = 10000` 또는 `0`(전체)
- checkpoint 경로는 `edit_tagger_v2`처럼 분리